# High-Order Kuramoto Network for Multiple Sequences; implementation of dense associative memory (Krotov)

This notebook tests d=3 high-order Kuramoto-type oscillator network 

$$\dot{\theta}_i
=
-\sin\theta_i
\sum_{\mu=1}^{P}
\xi_i^{\mu+1}
\left(
\frac{1}{N-1}
\sum_{j\neq i}
\xi_j^\mu
\cos\theta_j
\right)^d$$


For multiple sequences:

$$\dot{\theta}_i
=
-\sin\theta_i
\sum_{a=1}^{K}
\sum_{\mu=1}^{P_a}
\xi_i^{a,\mu+1}
\left(
\frac{1}{N-1}
\sum_{j\neq i}
\xi_j^{a,\mu}
\cos\theta_j
\right)^d$$

In [24]:

import numpy as np
import matplotlib.pyplot as plt
from numba import njit

np.set_printoptions(precision=3, suppress=True)


## 1. Model

For memory $\mu$, define the overlap of state with some pattern as

$$m_i^\mu(\boldsymbol{\theta})
=
\frac{1}{N-1}
\sum_{j\neq i}
\xi_j^\mu
\cos\theta_j$$




In [25]:
MEMORY_NAMES = ["A", "B", "C", "D", "E"]
A, B, C, D, E = range(5)

#note: I don't fully understand the K parameters initiliased.

DEFAULTS = dict(
    N=40,  # number of units / neurons in the associative memory network
    dt=0.02,  # integration timestep for the dynamical evolution
    T=55.0,  # total simulation time
    tau_s=10.0,  # time constant for the slow context trace that preserves previous memory activation
    frequency_std=0.03,  # heterogeneity in intrinsic frequencies, adding disorder to retrieval dynamics
    phase_noise=0.01,  # stochastic phase perturbations during evolution. 
    seed=3,  # random seed for reproducibility
    d = 3,

    #how to implement / add phase noise?

    cue_1_end=10.0,  # end time of the first cue period used to retrieve the initial history memory
    cue_B_end=22.0,  # end time of the second cue period that drives the intermediate memory B
)


In [ ]:
def make_memories(N=40, seed=DEFAULTS["seed"]):
    """Generate five random binary phase memories and the Hebbian coupling matrix."""
    rng = np.random.default_rng(seed)
    patterns = rng.choice([0.0, np.pi], size=(5, N))
    PATTERN_DICT = dict(zip(MEMORY_NAMES, patterns))
    return patterns, PATTERN_DICT


def overlaps(theta, patterns):
    """Return one overlap score per stored pattern for the current state."""
    c = np.cos(theta)
    weighted = patterns * c[None, :]
    total = weighted.sum(axis=1, keepdims=True)
    m_per_unit = (total - weighted) / (patterns.shape[1] - 1)
    return m_per_unit.mean(axis=1)


## The ODE Dynamics (Deterministic Forward Euler)

In [27]:


@njit
def simulate_ode_euler(theta_init, omega, patterns, d, dt, num_steps):
    """
    Deterministic Forward Euler simulation for oscillator dynamics.
    """
    N = theta_init.shape[0]
    P = patterns.shape[0] - 1
    
    # Pre-allocate history array
    history = np.empty((num_steps + 1, N), dtype=np.float64)
    theta = theta_init.copy()
    history[0] = theta
    
    # Pre-allocate static memory buffers
    cos_theta = np.empty(N, dtype=np.float64)
    sin_theta = np.empty(N, dtype=np.float64)
    S = np.empty(P, dtype=np.float64)
    
    for step in range(num_steps):
        # Precompute trigonometric arrays
        for i in range(N):
            cos_theta[i] = np.cos(theta[i])
            sin_theta[i] = np.sin(theta[i])
            
        # Precompute pattern-wide sums (S^mu)
        for mu in range(P):
            temp_sum = 0.0
            for j in range(N):
                temp_sum += patterns[mu, j] * cos_theta[j]
            S[mu] = temp_sum
            
        # Calculate derivatives and update states
        for i in range(N):
            V = 0.0
            for mu in range(P):
                inner_sum = S[mu] - patterns[mu, i] * cos_theta[i]
                h = (inner_sum / (N - 1)) ** d
                V += patterns[mu + 1, i] * h
            
            # Incorporate intrinsic frequency omega_i
            dtheta_i = omega[i] - sin_theta[i] * V
            
            # Deterministic update
            theta[i] = theta[i] + dt * dtheta_i
            
        history[step + 1] = theta.copy()
        
    return history

## The SDE Dynamics (Stochastic Euler-Maruyama)

In [28]:
@njit
def simulate_sde_maruyama(theta_init, omega, patterns, d, dt, num_steps, phase_noise):
    """
    Stochastic Euler-Maruyama simulation for oscillator dynamics with phase noise.
    """
    N = theta_init.shape[0]
    P = patterns.shape[0] - 1
    
    # Pre-allocate history array
    history = np.empty((num_steps + 1, N), dtype=np.float64)
    theta = theta_init.copy()
    history[0] = theta
    
    # Pre-allocate static memory buffers
    cos_theta = np.empty(N, dtype=np.float64)
    sin_theta = np.empty(N, dtype=np.float64)
    S = np.empty(P, dtype=np.float64)
    
    # Scaling factor for the Wiener process noise
    noise_scale = phase_noise * np.sqrt(dt)
    
    for step in range(num_steps):
        # Precompute trigonometric arrays
        for i in range(N):
            cos_theta[i] = np.cos(theta[i])
            sin_theta[i] = np.sin(theta[i])
            
        # Precompute pattern-wide sums (S^mu)
        for mu in range(P):
            temp_sum = 0.0
            for j in range(N):
                temp_sum += patterns[mu, j] * cos_theta[j]
            S[mu] = temp_sum
            
        # Calculate derivatives, add noise, and update states
        for i in range(N):
            V = 0.0
            for mu in range(P):
                inner_sum = S[mu] - patterns[mu, i] * cos_theta[i]
                h = (inner_sum / (N - 1)) ** d
                V += patterns[mu + 1, i] * h
            
            dtheta_i = omega[i] - sin_theta[i] * V
            
            # Stochastic update: Deterministic drift + Stochastic diffusion
            random_shock = np.random.randn()
            theta[i] = theta[i] + (dt * dtheta_i) + (noise_scale * random_shock)
            
        history[step + 1] = theta.copy()
        
    return history

In [29]:
def run_trial(config):

    seed=config["seed"]

    # xi corresponds to patterns

    mode = input("'ode' or 'sde') ").strip().lower()

    if mode == "":
        mode = "ode"

    """
    Coordinates a single simulation trial: initializes states, executes 
    the numerical integration (ODE or SDE), and tracks overlaps over time.
    
    Parameters:
    -----------
    config : dict
        Dictionary containing network parameters (e.g., DEFAULTS).
    xi : np.ndarray
        Pattern matrix of shape (P+1, N).
    mode : str
        The simulation style to run: 'ode' or 'sde'.
    seed : int, optional
        Random seed for reproducibility.
        
    Returns:
    --------
    results : dict
        Contains 'time', 'theta_history', and 'overlap_history'.
    """

        
    # 1. Extract parameters from config
    N = config['N']
    dt = config['dt']
    T = config['T']
    frequency_std = config['frequency_std']
    phase_noise = config['phase_noise']
    
    num_steps = int(T / dt)
    time_vector = np.linspace(0, T, num_steps + 1)

    xi, _ = make_memories()
    
    # 2. Initialize network states
    theta_0 = np.random.uniform(-np.pi, np.pi, N)
    omega = np.random.normal(0.0, frequency_std, N)
    d = config['d']  # Dynamic exponent parameter
    
    # 3. Execute the designated simulation
    if mode == 'ode':
        theta_history = simulate_ode_euler(
            theta_init=theta_0, omega=omega, patterns=xi, d=d, dt=dt, num_steps=num_steps
        )
    elif mode == 'sde':
        theta_history = simulate_sde_maruyama(
            theta_init=theta_0, omega=omega, patterns=xi, d=d, dt=dt, 
            num_steps=num_steps, phase_noise=phase_noise
        )
    else:
        raise ValueError("Mode must be either 'ode' or 'sde'")
        
    # 4. Compute overlaps over the entire time series
    sample_overlap = overlaps(theta_0, xi)
    P_elements = sample_overlap.shape[0]
    overlap_history = np.empty((num_steps + 1, P_elements))
    
    # Populate overlap values for each recorded time step
    for step in range(num_steps + 1):
        overlap_history[step] = overlaps(theta_history[step], xi)
        
    # 5. Package data
    results = {
        "time": time_vector,
        "theta_history": theta_history,
        "overlap_history": overlap_history
    }
    
    return results

In [31]:

trial_data = run_trial(DEFAULTS)

patterns, _ = make_memories()

P = patterns.shape[0] - 1
plt.figure(figsize=(10, 5))
for mu in range(P + 1):
    plt.plot(trial_data['time'], trial_data['overlap_history'][:, mu], label=f"Pattern {mu}")

# Visually mark your cue periods from the configuration dict
plt.axvline(x=DEFAULTS['cue_1_end'], color='blue', linestyle='--', label='Cue 1 End')
plt.axvline(x=DEFAULTS['cue_B_end'], color='red', linestyle='--', label='Cue B End')

plt.title("Evolution of Memory Pattern Overlaps Across Cue Windows")
plt.xlabel("Time (t)")
plt.ylabel("Overlap Magnitude")
plt.legend()
plt.grid(True)
plt.show()

ValueError: could not broadcast input array from shape (5,40) into shape (5,)